# Results: noise robustness and MZI vs tritter comparison

This notebook interprets the phase 1, 1bis, 2 and 3 results of the project
(see `docs/PLAN.md` and the companion explanation document for the theory and
the plan). All numbers come from the seed-0 runs of July 2026, produced by the
scripts in `src/` on the pinned stack (Perceval 1.2.3, MerLin 0.4.0).

The `runs/` directory is not versioned. To re-create everything from a fresh
clone (about one hour on a laptop CPU, fixed seeds, byte-reproducible):

```
python src/train_noise_grid.py
python src/mismatch_matrix.py
python src/compare_meshes.py
python src/train_financial.py
```

The key figures and CSV summaries these scripts write into `figures/` are
versioned, so the images below render even without re-running anything.


## Phase 1 recap: the pipeline learns

The committed baseline (MZI mesh, 6 modes, 3 photons, `two_gaussians` target,
800 steps) reaches the real-vs-real MMD floor and covers both modes of the
mixture. Nothing new here, it is the sanity check the rest builds on.

![](../figures/mmd_curve_two_gaussians.png)
![](../figures/scatter_two_gaussians.png)


## Phase 2: noise grid and the mismatch matrix

Five profiles were trained (P0 clean; P1, P2, P3 with indistinguishability
0.95, 0.90, 0.85; P4 with indistinguishability 0.95 plus transmittance 0.9),
same seed, same target, full Fock space forced everywhere. The matrix below
evaluates the model trained under the row profile inside the physics of the
column profile, weights moved with `noise.transfer_generator` (never a raw
`state_dict` load: P4 has 84 output states, the others 56).


In [ ]:
import json
from pathlib import Path

ROOT = Path("..")
r = json.loads((ROOT / "runs/noise_grid/mismatch_mzi.json").read_text())
profiles = r["profiles"]
w = 10
header = "train\\eval".ljust(w) + "".join(p.rjust(w) for p in profiles)
print(header)
for a in profiles:
    row = "".join(f"{r['matrix'][a][b]['mmd_mean']:.4f}".rjust(w) for b in profiles)
    print(a.ljust(w) + row)
print()
print(f"diagonal mean  {r['diag_mean']:.5f}")
print(f"off-diag mean  {r['offdiag_mean']:.5f}")
print(f"floor          {r['floor_mean']:.5f} +- {r['floor_std']:.5f}")
print(f"suspect_bug    {r['suspect_bug']}")


![](../figures/mismatch_heatmap_mzi.png)
![](../figures/mmd_final_vs_indistinguishability_mzi.png)

### Reading the matrix

Three things stand out, and they are not the ones we expected to matter most.

1. **Partial distinguishability barely hurts on this target.** The matched
   (diagonal) MMD rises only from 0.0052 (P0) to 0.0060 (P3) while the floor
   sits at 0.0033 +- 0.0013. Pushing indistinguishability from 1.0 down to
   0.85 costs almost nothing here. Frankly, the two-Gaussian target is
   probably too easy to stress the interference resource: the adapter can
   compensate a mildly blurred Fock distribution.

2. **Losses are the real mismatch axis.** Every model trained without losses
   collapses when evaluated under P4 (0.031 to 0.037, six to eight times its
   diagonal). The mechanism is structural: with transmittance 0.9 and three
   photons, a fraction 1 - 0.9^3 = 27% of the events lose at least one
   photon, and a loss-blind model has zero weight on all 28 low-photon
   states, so a quarter of the probability mass lands where the adapter was
   never trained.

3. **Training under the deployment noise repairs it.** The P4-diagonal cell
   (0.0039) is the best of the whole matrix, at the floor. The generator
   learns to use the enlarged output space. Even the reverse transfer
   (P4-trained, evaluated clean, 0.0062) only degrades mildly: the 56
   three-photon columns survive the move.

The acceptance criterion of the plan (diagonal at least as good as
off-diagonal on average) holds: 0.0052 vs 0.0112, `suspect_bug` is false.


## Phase 3: MZI vs tritter at fair budget

Both meshes go through the identical route (raw Perceval circuit, sandwich
structure, explicit input state |1,0,1,0,1,0>, FOCK forced): 60 trainable
phases for the MZI Clements mesh, 56 for the 12-layer tritter mesh, with
measured Jacobian ranks 50 vs 49 (`tests/test_review_findings.py`), so the
budget is fair in the effective sense, not only nominally.

A consistency note worth recording: the route-2 MZI reproduces the phase 2
route-1 (CircuitBuilder) mismatch matrix **bit for bit** (max cell difference
0.0). Same topology, same parameter order, same seed, same trajectory. Two
independent construction paths agreeing exactly is a nice free cross-check
of the raw-circuit path.


In [ ]:
import csv

rows = list(csv.DictReader(open(ROOT / "figures/final_table.csv")))
print(f"{'profile':<9}{'mzi':>10}{'tritter':>10}   better")
for p in ["P0", "P1", "P2", "P3", "P4"]:
    m = {row["mesh"]: float(row["mmd_mean"]) for row in rows if row["profile"] == p}
    better = min(m, key=m.get)
    print(f"{p:<9}{m['mzi']:>10.5f}{m['tritter']:>10.5f}   {better}")
print(f"floor    {float(rows[0]['floor_mean']):>10.5f}")


![](../figures/ranking_vs_noise.png)

### The answer to the research question

**The ranking is stable under partial distinguishability and it inverts
under losses.**

- On P0 through P3 the tritter mesh is consistently better (0.0040 to 0.0044
  against 0.0052 to 0.0060 for the MZI). Same direction on all four
  profiles, with four fewer trainable phases.
- On P4 the ranking flips: the MZI lands at the floor (0.0039) while the
  tritter degrades to 0.0058. The tritter's P4 row of its own mismatch
  matrix is also the weakest (P4-trained evaluated clean: 0.0070 to 0.0074
  against 0.0062 for the MZI).

Honest caveats before anyone over-reads this: single training seed, and the
per-cell evaluation std (0.001 to 0.003) is of the same order as the gaps,
so each individual comparison is a one-sigma statement. What gives it some
weight is the consistency of the direction across profiles, not any single
cell. Seed replication is the obvious next step before claiming more. A flat
result would also have been reportable; what we actually see is a
noise-dependent ranking, which is the more interesting outcome for the
question we asked.


In [ ]:
import sys

sys.path.insert(0, str((ROOT / "src").resolve()))

import matplotlib.pyplot as plt
import torch

from data import DATASETS
from mismatch_matrix import build_generator_from_log
from noise import make_noise  # noqa: F401  (imported for readers extending this cell)

# Matched-profile scatters rebuilt from the saved runs: same class and noise
# as training, so a plain state_dict load is safe here.
picks = [("mzi", "P0"), ("tritter", "P0"), ("tritter", "P4")]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
target = DATASETS["two_gaussians"](2000, seed=1)
for ax, (mesh, profile) in zip(axes, picks):
    run = ROOT / "runs/mesh_compare" / mesh / profile
    log = json.loads((run / "log.json").read_text())
    gen = build_generator_from_log(log, profile)
    gen.load_state_dict(torch.load(run / "model.pt", weights_only=True))
    gen.eval()
    with torch.no_grad():
        fake = gen(2000, generator=torch.Generator().manual_seed(1))
    ax.scatter(target[:, 0], target[:, 1], s=4, alpha=0.3, label="target")
    ax.scatter(fake[:, 0], fake[:, 1], s=4, alpha=0.3, label="generated")
    ax.set_title(f"{mesh} / {profile}")
    ax.set_aspect("equal")
    ax.legend()
fig.tight_layout()
plt.show()


## Phase 1bis: heavy-tailed 1D target

Without market data in the offline environment, the target is a standardized
Student-t with 4 degrees of freedom, the classic stand-in for daily
log-returns (leptokurtic, tail exponent in the empirical ballpark).
Standardizing to unit variance keeps the default MMD bandwidths meaningful
(review finding F5) and does not change the tail shape.


In [ ]:
print((ROOT / "figures/tails_log_returns.csv").read_text())


![](../figures/hist_log_returns.png)
![](../figures/qq_log_returns.png)

The bulk is captured well: the 5%, 50% and 95% quantiles match within a few
percent. The tails are not: at the 0.1% / 99.9% level the target reaches
-4.85 / +4.86 while the generator stops near -2.45 / +2.33. The QQ plot
flattens symmetrically at both ends, the signature of a generator with an
effectively bounded output range: the sample is an affine map of a 56-bin
probability vector driven by tanh-bounded angles, so extreme values would
need weights the MMD never asks for. With bandwidths of 0.1 and larger, the
0.1% tail mass contributes almost nothing to the loss. This was the expected
outcome and we report it as a result, not a failure: a generator at this
scale does not capture fat tails. Plausible follow-ups: adding sub-0.1
bandwidths, a heavy-tailed latent instead of the Gaussian, or an explicit
quantile penalty.


## Limitations and what comes next

- **Single seed.** Every number above is one training run (seed 0). The
  inter-mesh gaps are about one evaluation sigma; the mesh ranking needs 3
  to 5 seeds before it is more than suggestive.
- **Easy target.** `two_gaussians` trains to the floor even under P3 noise;
  the ring target or a harder mixture would separate the architectures
  better.
- **Estimator bias.** The MMD is the biased V-statistic at fixed batch 256;
  values are only comparable at that batch size. The real-vs-real floor
  (0.0033 +- 0.0013 over 16 disjoint pairs) is the reference, and several
  reported gaps are of the same order as its std.
- **No quantum advantage is claimed anywhere.** This is a characterization
  of where a small photonic generative layer breaks under realistic noise.
- **Phase 4** (inference of the best model on Quandela hardware, treating
  the QPU as an unknown noise profile in exactly this mismatch framework)
  is pending cloud access; the `.env` token mechanism is already in the
  README.
